# 🧠 Real-Time Stock Sentiment Consumer (FinBERT + MongoDB)

This Jupyter Notebook acts as the AI enrichment engine in our real-time streaming data pipeline. It consumes raw stock news messages from Apache Kafka, executes local financial sentiment classification using **ProsusAI/finbert**, and stores the enriched predictions into MongoDB.

---

### 🤖 Local FinBERT Execution (No External APIs, No Ollama):
- **Model**: `ProsusAI/finbert` (a BERT model fine-tuned specifically on financial phrasebanks and market communications).
- **Library**: Hugging Face `transformers` and `torch`.
- **Local Inference**: Loaded via `pipeline("text-classification", model="ProsusAI/finbert")`. On the first run, the model weights (~438MB) are downloaded directly from Hugging Face Hub to your local cache (`~/.cache/huggingface/hub`). Subsequent runs load 100% locally from disk.
- **Zero Third-Party APIs / No Ollama**: All tokenization, tensor operations, and inference run entirely on your local CPU or GPU.

---

### 📊 Sentiment Computation & MongoDB Storage:
1. **Input**: News messages consumed from Kafka topic `stock-news` (containing `ticker`, `headline`, `timestamp`).
2. **Inference**: Each `headline` is processed through the local FinBERT pipeline.
3. **Extraction**:
   - `sentiment`: The predicted class label (`POSITIVE`, `NEGATIVE`, or `NEUTRAL`).
   - `confidence`: The model prediction probability score, rounded to **4 decimal places**.
4. **Database Storage**:
   - **Target**: MongoDB instance at `mongodb://localhost:27017/`.
   - **Database**: `StockDB`
   - **Collection**: `news_sentiment`
   - Every enriched document is inserted into `StockDB.news_sentiment`.
5. **Console Logging**:
   - Prints a concise log line for every processed news event: `[SENTIMENT] TICKER: HEADLINE`.

---

### 🚀 How to Run in JupyterLab:
1. Ensure the Kafka container and MongoDB container are active (`docker compose up -d`).
2. Ensure the kernel is set to **`Python (Stock Sentiment)`** (or your active `.venv`).
3. Run the code cell below.
4. Watch real-time FinBERT predictions enrich the streaming headlines as they are ingested from Kafka into MongoDB.

---

### ⏹️ How to Stop Safely:
- Click the **Square Stop Button ("Interrupt the kernel")** on the JupyterLab toolbar.
- Or use the top menu: **Kernel -> Interrupt Kernel**.
- The `KeyboardInterrupt` exception will cleanly commit consumer offsets, close the MongoDB connection, and exit safely.

In [ ]:
import os
import sys
import json
import warnings
from datetime import datetime, timezone
warnings.filterwarnings('ignore')
os.environ['HF_HUB_DISABLE_SYMLINKS_WARNING'] = '1'
os.environ['TRANSFORMERS_VERBOSITY'] = 'error'
from pymongo import MongoClient
from kafka import KafkaConsumer
from transformers import pipeline

# ==========================================
# Configuration
# ==========================================
KAFKA_BOOTSTRAP_SERVERS = 'localhost:9092'
KAFKA_TOPIC = 'stock-news'
KAFKA_GROUP_ID = 'finbert-sentiment-consumer-group'

def resolve_mongo_uri():
    if os.getenv('MONGO_URI'):
        return os.getenv('MONGO_URI')
    secrets_file = os.path.join('.streamlit', 'secrets.toml')
    if os.path.exists(secrets_file):
        try:
            with open(secrets_file, 'r') as f:
                for line in f:
                    if line.strip().startswith('MONGO_URI'):
                        return line.split('=', 1)[1].strip().strip('"').strip("'")
        except Exception:
            pass
    return 'mongodb://localhost:27017/'

MONGO_URI = resolve_mongo_uri()
MONGO_DB = 'StockDB'
MONGO_COLLECTION = 'news_sentiment'
LOGS_COLLECTION = 'pipeline_logs'

MODEL_NAME = 'ProsusAI/finbert'
MAX_MESSAGES = int(os.getenv('MAX_CONSUMER_MESSAGES', '0'))

def run_consumer():
    """Main function to consume Kafka messages, classify with FinBERT, and store in MongoDB."""
    print('=' * 80)
    print('Starting Financial Sentiment Consumer (FinBERT + MongoDB)')
    print('=' * 80)

    # 1. Load FinBERT locally via Hugging Face Transformers
    print(f'\n[1/3] Loading local FinBERT model ("{MODEL_NAME}")...')
    print('      (Runs entirely on local CPU/GPU via PyTorch; no external APIs / no Ollama)')
    try:
        classifier = pipeline('text-classification', model=MODEL_NAME)
        print('      FinBERT pipeline loaded successfully into local memory.')
    except Exception as e:
        print(f'      [Error] Failed to load model "{MODEL_NAME}": {e}')
        return

    # 2. Connect to MongoDB
    masked_uri = MONGO_URI.split('@')[-1] if '@' in MONGO_URI else MONGO_URI
    print(f'\n[2/3] Connecting to MongoDB cluster ({masked_uri})...')
    try:
        mongo_client = MongoClient(MONGO_URI, serverSelectionTimeoutMS=5000)
        mongo_client.admin.command('ping')
        db = mongo_client[MONGO_DB]
        collection = db[MONGO_COLLECTION]
        logs_col = db[LOGS_COLLECTION]
        print(f'      Connected successfully to database "{MONGO_DB}", collection "{MONGO_COLLECTION}".')
    except Exception as e:
        print(f'      [Error] Could not connect to MongoDB: {e}')
        print('      Please ensure MongoDB container is running ("docker compose up -d").')
        return

    # 3. Connect to Kafka Consumer
    print(f'\n[3/3] Initializing Kafka consumer on topic "{KAFKA_TOPIC}"...')
    try:
        consumer = KafkaConsumer(
            KAFKA_TOPIC,
            bootstrap_servers=KAFKA_BOOTSTRAP_SERVERS,
            auto_offset_reset='earliest',
            enable_auto_commit=True,
            group_id=KAFKA_GROUP_ID,
            value_deserializer=lambda m: json.loads(m.decode('utf-8'))
        )
        print('      Kafka consumer ready. Waiting for incoming stock news stream...')
        print('      Press Kernel -> Interrupt (or Stop button) to halt.')
        print('-' * 80)
    except Exception as e:
        print(f'      [Error] Failed to connect Kafka consumer: {e}')
        print(f'      Please ensure Kafka broker is running at {KAFKA_BOOTSTRAP_SERVERS}.')
        mongo_client.close()
        return

    # 4. Stream Processing Loop
    record_count = 0
    try:
        for message in consumer:
            payload = message.value
            headline = payload.get('headline', '')
            ticker = payload.get('ticker', 'UNKNOWN')
            timestamp = payload.get('timestamp', datetime.now(timezone.utc).isoformat())

            # Local FinBERT classification
            inference_result = classifier(headline)[0]
            sentiment_label = inference_result.get('label', 'NEUTRAL').upper()
            confidence_score = round(float(inference_result.get('score', 0.0)), 4)

            enriched_record = {
                'ticker': ticker,
                'headline': headline,
                'timestamp': timestamp,
                'sentiment': sentiment_label,
                'confidence': confidence_score
            }

            # Persist into MongoDB
            collection.insert_one(enriched_record)
            record_count += 1

            # Log to MongoDB Atlas pipeline_logs (visible live on App dashboard)
            try:
                now_str = datetime.now(timezone.utc).strftime('%H:%M:%S.%f')[:-3]
                logs_col.insert_one({
                    'time': now_str,
                    'iso_time': timestamp,
                    'level': 'FINBERT',
                    'component': 'KAFKA_CONSUMER',
                    'message': f'[{sentiment_label}] {ticker}: "{headline[:55]}..." (conf: {confidence_score:.4f})'
                })
            except Exception:
                pass

            # Print log line
            print(f'[{sentiment_label}] {ticker}: {headline} (confidence: {confidence_score:.4f})')
            sys.stdout.flush()

            if MAX_MESSAGES > 0 and record_count >= MAX_MESSAGES:
                print(f'[Consumer] Completed processing {MAX_MESSAGES} messages. Exiting loop cleanly.')
                break

    except KeyboardInterrupt:
        print(f'\n[Consumer] Interrupted by user. Total messages processed: {record_count}.')
    except Exception as err:
        print(f'\n[Consumer Error] Stream processing exception: {err}')
    finally:
        print('[Consumer] Closing Kafka consumer and MongoDB connections...')
        consumer.close()
        mongo_client.close()
        print('[Consumer] Shutdown complete. Resources released cleanly.')

run_consumer()
